In [ ]:
import torch
import time
import os
import torchvision.models as models
from torchvision.io import read_image
from torchvision import transforms
from torchvision.models import AlexNet_Weights, VGG16_Weights, ResNet50_Weights
from tabulate import tabulate

In [ ]:
path = "/content/drive/MyDrive/Colab Notebooks/images/"
images = [f for f in os.listdir(path) if f.endswith(('.png', '.jpg', '.JPEG'))]

In [ ]:
def preprocess_image(img):
    preprocess = transforms.Compose([
        transforms.Resize(256), # нормализация
        transforms.CenterCrop(224), # Обрезать фото для Vgg16,ResNet, AlexNet
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    return preprocess(img / 255.0)

In [ ]:
def predict_top5(model, preprocessed_img, categories, result):
    start_time = time.time()
    prediction = model(preprocessed_img.unsqueeze(0)).squeeze(0).softmax(0) #
    top5_accuracy, top5_id = torch.topk(prediction, 5)
    end_time = time.time()

    print(f"{model_name}: ")
    for i in range(5):
        class_id = top5_id[i].item()
        score = top5_accuracy[i].item()
        print(f"  {i+1}. {categories[class_id]} ({class_id}): {score:.2%}")
    print(f"Running time: {end_time - start_time}")
    top1 = categories[top5_id[0].item()]
    return top1.lower() == result.lower(),

In [ ]:
models_weights = {
    "resnet50": (models.resnet50, ResNet50_Weights.DEFAULT),
    "alexnet": (models.alexnet, AlexNet_Weights.DEFAULT),
    "vgg16": (models.vgg16, VGG16_Weights.DEFAULT),
}

In [ ]:
models = {}
for name, (model_func, weights) in models_weights.items():
    model = model_func(weights=weights) # инициализации модели c предварительным обученным весом
    models[name] = model.eval()

In [ ]:
# массив для хранения правильных прогнозов
results = []
correct_counts = {model_name: 0 for model_name in models.keys()}

In [ ]:
for image in images:
    img = read_image(os.path.join(path, image))
    preprocessed_img = preprocess_image(img)
    categories = weights.meta["categories"]
    result = os.path.splitext(image)[0].strip()

    print(f"\nImage: {image}")
    row = [image]
    for model_name, model in models.items():
        is_correct = predict_top5(model, preprocessed_img, categories, result)
        if is_correct: correct_counts[model_name] += 1
        row.append("Correct" if is_correct else "Wrong")

    results.append(row)

In [ ]:
headers = ["Image File"] + [model_name.capitalize() for model_name in models.keys()]
print("\nResults Table:")
print(tabulate(results, headers=headers, tablefmt="grid"))

In [ ]:
print("\nFinal Results:")
for model_name, correct_count in correct_counts.items():
    accuracy = (correct_count / len(images)) * 100
    print(f"{model_name.capitalize()} - Correct Top-1 Predictions: {correct_count}/{len(images)} ({accuracy:.2f}%)")